# 09 · Non-motor symptom network (Graphical Lasso)

Method adapted from the PIGD compendium (Technique 8). Estimate a sparse partial-correlation network of non-motor symptoms, with NP1PAIN as a node. Report pain's neighbors and the top betweenness-centrality hub. Compare baseline networks for DBS vs Never-DBS to see whether the topology of symptom linkage differs.

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(ggplot2); library(glasso); library(qgraph); library(igraph)
})

df <- readRDS(file.path(OUT_OBJ, "pain_long.rds"))
# Baseline = earliest visit per PATNO
base <- df %>% dplyr::arrange(PATNO, INFODT_orig) %>%
  dplyr::group_by(PATNO) %>% dplyr::slice_head(n = 1) %>% dplyr::ungroup()

node_vars <- c(
  "NP1PAIN","NP1SLPN","NP1SLPD","NP1FATG","NP1URIN",
  "NP1DPRS","NP1ANXS",
  "gds","stai","ess","rem",
  "scopa","updrs3_score","BMI","LEDD"
)
X0 <- base %>% dplyr::select(PATNO, will_receive_dbs, dplyr::all_of(node_vars))
cat("Baseline rows:", nrow(X0), "\n")
summary(X0 %>% dplyr::select(-PATNO, -will_receive_dbs))

Warning message:
“package ‘igraph’ was built under R version 4.5.2”


Baseline rows: 170 


    NP1PAIN          NP1SLPN         NP1SLPD          NP1FATG      
 Min.   :0.0000   Min.   :0.000   Min.   :0.0000   Min.   :0.0000  
 1st Qu.:0.0000   1st Qu.:0.000   1st Qu.:0.0000   1st Qu.:0.0000  
 Median :1.0000   Median :1.000   Median :1.0000   Median :1.0000  
 Mean   :0.9647   Mean   :1.129   Mean   :0.8588   Mean   :0.7765  
 3rd Qu.:1.0000   3rd Qu.:2.000   3rd Qu.:1.0000   3rd Qu.:1.0000  
 Max.   :4.0000   Max.   :4.000   Max.   :4.0000   Max.   :4.0000  
                                                                   
    NP1URIN          NP1DPRS          NP1ANXS            gds        
 Min.   :0.0000   Min.   :0.0000   Min.   :0.0000   Min.   : 0.000  
 1st Qu.:0.0000   1st Qu.:0.0000   1st Qu.:0.0000   1st Qu.: 1.000  
 Median :1.0000   Median :0.0000   Median :0.0000   Median : 2.000  
 Mean   :0.7765   Mean   :0.3588   Mean   :0.5294   Mean   : 2.685  
 3rd Qu.:1.0000   3rd Qu.:1.0000   3rd Qu.:1.0000   3rd Qu.: 4.000  
 Max.   :4.0000   Max.   :4.0000   Max.   

In [2]:
# Complete-case matrix for glasso (drop patients missing any node)
build_matrix <- function(d) {
  d %>% dplyr::select(dplyr::all_of(node_vars)) %>% tidyr::drop_na() %>% as.matrix()
}

# Fit glasso at a ρ that gives a sparse but connected network
fit_glasso <- function(M, rho = 0.12) {
  S <- stats::cor(M)
  g <- glasso::glasso(S, rho = rho)
  Theta <- g$wi
  colnames(Theta) <- rownames(Theta) <- colnames(M)
  # Partial correlation from Theta
  P <- -Theta / sqrt(outer(diag(Theta), diag(Theta)))
  diag(P) <- 0
  P
}

M_all  <- build_matrix(X0)
M_dbs  <- build_matrix(X0 %>% dplyr::filter(will_receive_dbs))
M_ctl  <- build_matrix(X0 %>% dplyr::filter(!will_receive_dbs))
cat("n complete-case — all:", nrow(M_all),
    "  DBS:", nrow(M_dbs), "  Never-DBS:", nrow(M_ctl), "\n")

P_all <- fit_glasso(M_all)
P_dbs <- fit_glasso(M_dbs)
P_ctl <- fit_glasso(M_ctl)

n complete-case — all: 104   DBS: 35   Never-DBS: 69 


In [3]:
# Pain's neighbors (partial correlations with NP1PAIN) in the all-patients network
pain_row <- tibble::tibble(
  partner = colnames(P_all),
  partial_cor = P_all["NP1PAIN", ]
) %>% dplyr::filter(partner != "NP1PAIN") %>%
  dplyr::arrange(dplyr::desc(abs(partial_cor)))
print(pain_row)
save_table(pain_row, "pain_partial_correlations_all")

# A tibble: 14 × 2
   partner      partial_cor
   <chr>              <dbl>
 1 scopa            0.172  
 2 NP1SLPN          0.124  
 3 rem              0.0970 
 4 NP1FATG          0.0598 
 5 gds              0.0498 
 6 stai             0.0438 
 7 LEDD             0.0395 
 8 NP1ANXS          0.0124 
 9 ess              0.00874
10 NP1SLPD          0      
11 NP1URIN          0      
12 NP1DPRS          0      
13 updrs3_score     0      
14 BMI              0      


In [4]:
# Network centrality (igraph) — which node is the hub?
centrality_of <- function(P, threshold = 0.05, label) {
  A <- abs(P); A[A < threshold] <- 0
  g <- igraph::graph_from_adjacency_matrix(A, mode = "undirected", weighted = TRUE, diag = FALSE)
  tibble::tibble(
    network = label,
    node = colnames(P),
    degree = igraph::degree(g),
    strength = igraph::strength(g),
    betweenness = igraph::betweenness(g, weights = 1/(igraph::E(g)$weight + 1e-6))
  )
}
cent <- dplyr::bind_rows(
  centrality_of(P_all, label = "All"),
  centrality_of(P_dbs, label = "DBS"),
  centrality_of(P_ctl, label = "Never-DBS")
)
cent_wide <- cent %>% dplyr::select(network, node, strength) %>%
  tidyr::pivot_wider(names_from = network, values_from = strength)
print(cent %>% dplyr::filter(network == "All") %>% dplyr::arrange(dplyr::desc(betweenness)))
save_table(cent, "network_centrality")

Warning message:
“The `adjmatrix` argument of `graph_from_adjacency_matrix()` must be symmetric
with mode = "undirected" as of igraph 1.6.0.
ℹ Use mode = "max" to achieve the original behavior.”


# A tibble: 15 × 5
   network node         degree strength betweenness
   <chr>   <chr>         <dbl>    <dbl>       <dbl>
 1 All     scopa             8    1.12           32
 2 All     gds               6    1.07           26
 3 All     NP1SLPD           4    0.664          11
 4 All     NP1FATG           6    0.607           7
 5 All     NP1SLPN           6    0.589           4
 6 All     LEDD              3    0.327           4
 7 All     NP1PAIN           4    0.453           2
 8 All     ess               4    0.466           2
 9 All     rem               5    0.471           1
10 All     NP1URIN           1    0.379           0
11 All     NP1DPRS           3    0.539           0
12 All     NP1ANXS           5    0.552           0
13 All     stai              3    0.663           0
14 All     updrs3_score      2    0.187           0
15 All     BMI               0    0               0


In [5]:
# Plot the all-patients network
png(file.path(OUT_FIG, "Fig16_symptom_network_all.png"), width = 1800, height = 1400, res = 200)
qgraph::qgraph(P_all, layout = "spring", theme = "classic",
               labels = colnames(P_all), label.cex = 1.1,
               title = "Non-motor symptom partial-correlation network (all patients)")
invisible(dev.off())

png(file.path(OUT_FIG, "Fig17_symptom_network_dbs.png"), width = 1800, height = 1400, res = 200)
qgraph::qgraph(P_dbs, layout = "spring", theme = "classic",
               labels = colnames(P_dbs), label.cex = 1.1,
               title = "Network — future-DBS patients at baseline")
invisible(dev.off())

png(file.path(OUT_FIG, "Fig18_symptom_network_never_dbs.png"), width = 1800, height = 1400, res = 200)
qgraph::qgraph(P_ctl, layout = "spring", theme = "classic",
               labels = colnames(P_ctl), label.cex = 1.1,
               title = "Network — Never-DBS patients at baseline")
invisible(dev.off())
cat("Network plots saved.\n")

Network plots saved.


In [6]:
# Save partial-correlation matrices
save_object(list(all = P_all, dbs = P_dbs, ctl = P_ctl), "partial_correlation_matrices")
cat("Saved partial_correlation_matrices.rds\n")

Saved partial_correlation_matrices.rds
